# 함수 호출
- 대규모 언어 모델을 외부 도구에 연결하는 방법  

- API 호출에서 함수를 설명하고 모델이 하나 이상의 함수를 호출하기 위한 인수가 포함된 JSON 개체를 출력하도록 지능적으로 선택하도록 할 수 있습니다. Chat Completions API는 함수를 호출하지 않습니다. 대신 모델은 코드에서 함수를 호출하는 데 사용할 수 있는 JSON을 생성합니다.
- 모델은 함수를 호출해야 하는 시기를 감지하고 함수를 준수하는 JSON으로 응답하도록 훈련되었습니다. 이 기능에는 잠재적인 위험도 따릅니다. 사용자를 대신하여 세상에 영향을 미치는 조치(이메일 전송, 온라인 게시, 구매 등)를 수행하기 전에 사용자 확인 흐름을 구축하는 것이 좋습니다.

### 채팅 모델로 함수를 호출하는 방법

함수 호출을 위한 기본 단계는 다음과 같습니다:

1. **모델 호출**: 사용자 질의와 함께 함수 집합을 정의하여 모델을 호출합니다. 함수 목록을 `tools` 파라미터에 제공합니다.


2. **모델의 함수 호출 선택**: 모델은 하나 이상의 함수를 호출할 수 있습니다. 이 경우, 모델의 응답 내용은 사용자가 정의한 스키마에 따른 JSON 객체 문자열이 됩니다. 


3. **JSON 파싱 및 함수 호출**: 코드에서 문자열을 JSON으로 파싱하고, 제공된 인수가 있는 경우 함수를 호출합니다.


4. **모델 재호출 및 결과 요약**: 함수 응답을 새로운 메시지로 추가하여 모델을 다시 호출합니다. 모델이 결과를 사용자에게 요약하여 전달합니다.

이 단계들은 사용자 질의에 따라 적절한 함수를 선택하고, 해당 함수의 응답을 처리하여 최종 결과를 사용자에게 제공하는 과정을 포함합니다. 이를 통해 사용자는 더 나은 응답을 받을 수 있습니다.

<img src="https://i.imgur.com/fR4FuHa.png" width=400 />

### 함수 정의 (Defining Functions)
- 각 API 요청의 tools 매개변수 내에서 함수를 정의할 수 있습니다.
- 함수는 스키마(Schema) 를 통해 정의되며, 모델이 함수의 목적과 입력 인수를 이해할 수 있도록 돕습니다.
스키마는 다음과 같은 필드로 구성됩니다.
| 필드        | 설명                                      |
|------------|------------------------------------------|
| **name**   | 함수의 이름 (예: `get_weather`)          |
| **description** | 함수가 언제, 어떻게 사용되는지 설명  |
| **parameters**  | 함수의 입력 인수를 정의하는 JSON 스키마 |


In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv()) # read local .env file

True

In [2]:
from openai import OpenAI
import json

client = OpenAI()

Model = "gpt-4.1-nano"

In [3]:
# 예제 더미 함수로 하드 코딩된 동일한 날씨 정보를 반환합니다.
# 실제 환경에서는 백엔드 API 또는 외부 API가 될 수 있습니다.
def get_current_weather(location, unit="celsius"):
    """지정된 위치의 현재 날씨를 가져옵니다."""
    if "tokyo" in location.lower():
        return json.dumps({"location": "Tokyo", "temperature": "10", "unit": unit})
    elif "san francisco" in location.lower():
        return json.dumps({"location": "San Francisco", "temperature": "22.2", "unit": unit})
    elif "paris" in location.lower():
        return json.dumps({"location": "Paris", "temperature": "22", "unit": unit})
    else:
        return json.dumps({"location": location, "temperature": "unknown"})

get_current_weather("Paris")

'{"location": "Paris", "temperature": "22", "unit": "celsius"}'

In [4]:
#함수 집합 정의
tools = [
    {
        "type": "function",   # 도구의 유형을 함수로 설정
        "name": "get_current_weather",   # 함수 이름
        "description": "지정된 위치의 현재 날씨를 가져오는 함수.",    # 함수 설명
        "parameters": {
            "type": "object",    # 파라미터의 유형을 객체로 설정
            "properties": {
                "location": {
                    "type": "string",   # location 파라미터의 데이터 유형을 문자열로 설정
                    "description": "날씨를 알고 싶은 도시와 주 이름. 예: San Francisco, CA",  # location 파라미터 설명
                },
            },
            "required": ["location"],   # location 파라미터는 필수
        },
    }
]

In [5]:
# Step 1: 모델 호출: 사용자 질의와 함께 함수 집합을 정의하여 모델을 호출합니다. 함수 목록을 tools 파라미터에 제공합니다.
# Step 2: 모델은 함수를 호출하기로 결정합니다. 모델은 이름 과 입력 인수를 반환합니다 . 
# 이 경우, 모델의 응답 내용은 사용자가 정의한 스키마에 따른 JSON 객체 문자열이 됩니다.
response = client.responses.create(
    model=Model,
    input=[{"role": "user", "content": "샌프란시스코, 도쿄, 파리의 날씨는 어떻습니까?"}],
    tools=tools
)

response.output

[ResponseFunctionToolCall(arguments='{"location":"San Francisco"}', call_id='call_sFUVmteFUkrc1zmfvDluZ6hA', name='get_current_weather', type='function_call', id='fc_6844d2f9957881909219498f6c4388980943183a03730452', status='completed'),
 ResponseFunctionToolCall(arguments='{"location":"Tokyo"}', call_id='call_15Sbjddh0Oeb1s9Hz3hRfmbh', name='get_current_weather', type='function_call', id='fc_6844d2f9b7388190ae83b2cff1ba7ebe0943183a03730452', status='completed'),
 ResponseFunctionToolCall(arguments='{"location":"Paris"}', call_id='call_DUrTAKL21XPYHvZSDTuCI2kF', name='get_current_weather', type='function_call', id='fc_6844d2f9cb7881909b7d17af41f75c200943183a03730452', status='completed')]

In [6]:
# step 3: 함수 코드 실행 – 모델의 응답을 구문 분석하고 함수 호출을 처리합니다 .
input_messages = [{"role": "user", "content": "샌프란시스코, 도쿄, 파리의 날씨는 어떻습니까?"}]

for i in range(len(response.output)):
    tool_call = response.output[i]
    args = json.loads(tool_call.arguments)
    result = get_current_weather(args['location'])
    input_messages.append(tool_call)    # append model's function call message
    input_messages.append({             # append result message
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": str(result)
    })

input_messages

[{'role': 'user', 'content': '샌프란시스코, 도쿄, 파리의 날씨는 어떻습니까?'},
 ResponseFunctionToolCall(arguments='{"location":"San Francisco"}', call_id='call_sFUVmteFUkrc1zmfvDluZ6hA', name='get_current_weather', type='function_call', id='fc_6844d2f9957881909219498f6c4388980943183a03730452', status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_sFUVmteFUkrc1zmfvDluZ6hA',
  'output': '{"location": "San Francisco", "temperature": "22.2", "unit": "celsius"}'},
 ResponseFunctionToolCall(arguments='{"location":"Tokyo"}', call_id='call_15Sbjddh0Oeb1s9Hz3hRfmbh', name='get_current_weather', type='function_call', id='fc_6844d2f9b7388190ae83b2cff1ba7ebe0943183a03730452', status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_15Sbjddh0Oeb1s9Hz3hRfmbh',
  'output': '{"location": "Tokyo", "temperature": "10", "unit": "celsius"}'},
 ResponseFunctionToolCall(arguments='{"location":"Paris"}', call_id='call_DUrTAKL21XPYHvZSDTuCI2kF', name='get_current_weather', type='function_ca

In [7]:
# step 4: 결과가 포함된 모델을 제공하여 최종 대응에 반영할 수 있습니다.
response_2 = client.responses.create(
    model=Model,
    input=input_messages,
    tools=tools,
)
print(response_2.output_text)

샌프란시스코의 현재 날씨는 약 22.2도입니다. 도쿄는 약 10도, 파리는 약 22도입니다.


### Free Weather API 이용
https://open-meteo.com/

In [8]:
import requests

# 현재의 온도 가져오기
def get_weather(latitude, longitude):
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m")
    data = response.json()
    return data['current']['temperature_2m']

# 서울의 위도, 경도
get_weather(37.56667, 126.97806)

20.9

In [9]:
import json

tools = [{
    "type": "function",
    "name": "get_weather",
    "description": "제공된 좌표의 현재 기온을 섭씨(Celsius) 단위로 가져오세요.",
    "parameters": {
        "type": "object",
        "properties": {
            "latitude": {"type": "number"},
            "longitude": {"type": "number"}
        },
        "required": ["latitude", "longitude"],
        "additionalProperties": False
    },
    "strict": True   # JSON 스키마의 엄격한 준수(strict mode)를 적용
    }]

response = client.responses.create(
    model="gpt-4.1-mini",
    input=[{"role": "user", "content": "오늘 서울의 기온이 어때?"}],
    tools=tools,
)

response.output

[ResponseFunctionToolCall(arguments='{"latitude":37.5665,"longitude":126.978}', call_id='call_xqgIpFlLKHFri4S0yY4mbWNx', name='get_weather', type='function_call', id='fc_6844d2ffa880819093dc6f04939820f70aa29a6b709965df', status='completed')]

In [10]:
# 함수 호출 실행
tool_call = response.output[0]
args = json.loads(tool_call.arguments)

result = get_weather(args['latitude'], args['longitude'])
result

20.9

In [11]:
# 함수 호출 실행 결과를 추가합니다.
input_messages = [{"role": "user", "content": "오늘 서울의 기온이 어때?"}]
input_messages.append(tool_call)
input_messages.append({
    "type": "function_call_output",
    "call_id": tool_call.call_id,
    "output": str(result)
})

response_2 = client.responses.create(
    model=Model,
    input=input_messages,
    tools=tools,
)

response_2.output_text

'오늘 서울의 기온은 약 20.9도입니다.'

### Email 보내기

In [12]:
# email 생성 함수
def send_email(to, subject, body):
    from email.utils import formatdate
    from email.mime.multipart import MIMEMultipart
    from email.header import Header
    # 메일 제목과 내용을 설정하는 모듈
    from email.mime.text import MIMEText
    
    msg = MIMEMultipart()

    port = 465  # For SSL
    smtp_server = "smtp.gmail.com"
    sender_email = "xxcompany@xxx.com" 
    receiver_email = to

    msg['From'] = sender_email 
    msg['To'] = receiver_email 
    msg['Date'] = formatdate(localtime=True)

    msg['Subject'] = Header(s=subject, charset='utf-8')
    body = MIMEText(body, _charset='utf-8')
    msg.attach(body)
        
    return f"{to} 에게 성공적으로 메일 발송"
    

send_email("test@naver.com", "test", "test")

'test@naver.com 에게 성공적으로 메일 발송'

In [13]:
# 이메일 보내기(send_email) 함수 호출 예제
tools = [{
    "type": "function",
    "name": "send_email",
    "description": "지정된 수신자에게 제목과 메시지를 포함한 이메일을 보내세요.",
    "parameters": {
        "type": "object",
        "properties": {
            "to": {
                "type": "string",
                "description": "수신자의 이메일 주소"
            },
            "subject": {
                "type": "string",
                "description": "이메일 제목 줄"
            },
            "body": {
                "type": "string",
                "description": "이메일 본문"
            }
        },
        "required": [
            "to",
            "subject",
            "body"
        ],
        "additionalProperties": False
    },
    "strict": True
}]

messages=[{"role": "user", 
           "content": "ilan@example.com 과 katia@example.com 에게 `안녕하세요`라는 내용을 담은 이메일을 보내줄 수 있나요?"}]

response = client.responses.create(
    model=Model,
    input=messages,
    tools=tools
)

response.output

[ResponseFunctionToolCall(arguments='{"to":"ilan@example.com","subject":"안녕하세요","body":"안녕하세요"}', call_id='call_yXXOGwwbu3VeYWF0sYdUX1vL', name='send_email', type='function_call', id='fc_6844d3030a44819580c68dbdf8492514079d807badf0c2be', status='completed'),
 ResponseFunctionToolCall(arguments='{"to":"katia@example.com","subject":"안녕하세요","body":"안녕하세요"}', call_id='call_h0Xw4s2JTLFdjVifFgeV52Gl', name='send_email', type='function_call', id='fc_6844d303373c81959f13149d4a0f94b9079d807badf0c2be', status='completed')]

In [14]:
def call_function(name, args):
    if name == "get_weather":
        return get_weather(**args)
    if name == "send_email":
        return send_email(**args)

In [15]:
#함수 호출 및 실행 결과 추가
input_messages = messages

for tool_call in response.output:
    if tool_call.type != "function_call":
        continue

    name = tool_call.name
    args = json.loads(tool_call.arguments)
    result = call_function(name, args)
    
    input_messages.append(tool_call)  # append model's function call message
    input_messages.append({
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": str(result)
    })

input_messages

[{'role': 'user',
  'content': 'ilan@example.com 과 katia@example.com 에게 `안녕하세요`라는 내용을 담은 이메일을 보내줄 수 있나요?'},
 ResponseFunctionToolCall(arguments='{"to":"ilan@example.com","subject":"안녕하세요","body":"안녕하세요"}', call_id='call_yXXOGwwbu3VeYWF0sYdUX1vL', name='send_email', type='function_call', id='fc_6844d3030a44819580c68dbdf8492514079d807badf0c2be', status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_yXXOGwwbu3VeYWF0sYdUX1vL',
  'output': 'ilan@example.com 에게 성공적으로 메일 발송'},
 ResponseFunctionToolCall(arguments='{"to":"katia@example.com","subject":"안녕하세요","body":"안녕하세요"}', call_id='call_h0Xw4s2JTLFdjVifFgeV52Gl', name='send_email', type='function_call', id='fc_6844d303373c81959f13149d4a0f94b9079d807badf0c2be', status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_h0Xw4s2JTLFdjVifFgeV52Gl',
  'output': 'katia@example.com 에게 성공적으로 메일 발송'}]

In [16]:
# 함수 호출 결과를 모델에 다시 보내기
response_2 = client.responses.create(
    model=Model,
    input=input_messages,
    tools=tools,
)

response_2.output_text

'이메일이 성공적으로 발송되었습니다. 추가로 도와드릴 일이 있나요?'

--------------------------
# 실습: 함수 호출 코드 작성


### 문제 1: 간단한 날씨 함수 호출 흐름 구성

**목표:**
다음 사용자 질문에 대해 모델이 `get_current_weather` 함수를 호출하고, 결과를 받아 최종 응답을 생성하도록 전체 흐름을 완성하세요.

**사용자 입력:**

> `"부산의 날씨가 어떤지 알려줘"`

**요구사항:**

* `tools` 정의
* `client.responses.create()` 호출 → 함수 호출 유도
* 함수 실행 결과를 포함한 메시지 리스트 구성
* 모델에게 최종 응답 요청

<details>
<summary>추가 조건</summary>

`get_current_weather` 함수는 "busan"에 대해 `"temperature": "24"`를 반환하도록 하드코딩하세요.

</details>

### 문제 2: Open-Meteo API를 이용한 실제 날씨 정보 조회 

**목표:**
사용자가 "서울 날씨 알려줘"라고 입력하면 모델이 위도/경도를 추론하고, `get_weather(latitude, longitude)` 함수를 호출한 뒤, 결과를 출력하도록 하세요.

**요구사항:**

* 서울의 위도: `37.56667`, 경도: `126.97806`
* `tools` 파라미터에 함수 스키마 정의
* JSON으로 파라미터가 잘 전달되었는지 확인
* `function_call_output` 메시지를 구성하여 최종 응답 요청

### 문제 3: 이메일 전송 함수 호출 (심화)

**목표:**
사용자가 다음 문장을 입력하면 모델이 `send_email()` 함수를 호출하도록 구성하세요.

> `"kim@example.com에게 제목은 '중요 공지', 내용은 '내일 회의가 연기되었습니다'라는 이메일을 보내줘"`

**요구사항:**

* `tools` 정의
* 호출된 함수의 arguments를 JSON으로 파싱하여 `send_email()`을 실행
* 최종 응답 지까지 완성

---